In [18]:
import json
from pathlib import Path
from typing import List, Dict
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma




In [8]:
CHUNKS_DIR = Path("../data/chunks/text")
CHROMA_DIR = Path("../data/chroma_db")


In [11]:
def load_chunks(path: Path) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

text_chunks = load_chunks(CHUNKS_DIR/ "docling_text_chunks.json")
table_chunks = load_chunks(CHUNKS_DIR/ "table_chunks.json")
ocr_chunks = load_chunks(CHUNKS_DIR/ "ocr_chunks.json")
image_chunks = load_chunks(CHUNKS_DIR/ "image_chunks.json")

print("Text chunks :", len(text_chunks))
print("Table chunks:", len(table_chunks))
print("OCR chunks  :", len(ocr_chunks))
print("Image chunks:", len(image_chunks))

Text chunks : 278
Table chunks: 65
OCR chunks  : 242
Image chunks: 51


In [12]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/bc/p_nnrzps7wdgbtdptyv7hg_h0000gn/T/ipykernel_26821/118603400.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/Users/sha/Developer/contextIQ/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
documents = []
metadatas = []
ids = []

for chunk in text_chunks + table_chunks + ocr_chunks:
    documents.append(chunk["content"])
    metadatas.append({
        "source": chunk["source"],
        "type": chunk["type"],
        "page": chunk.get("page"),
        "modality": chunk.get("modality", "text")
    })
    ids.append(chunk["chunk_id"])
print("total documents to embed:", len(documents))

total documents to embed: 585


In [19]:
vectordb = Chroma(
    persist_directory = str(CHROMA_DIR),
    embedding_function = embedding_model
)

/var/folders/bc/p_nnrzps7wdgbtdptyv7hg_h0000gn/T/ipykernel_26821/1463263660.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [20]:
vectordb.add_texts(
    texts = documents,
    metadatas = metadatas,
    ids = ids
)

vectordb.persist()
print("ChromaDB updated")

ChromaDB updated


/var/folders/bc/p_nnrzps7wdgbtdptyv7hg_h0000gn/T/ipykernel_26821/5021028.py:7: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [21]:
with open(CHROMA_DIR / "image_metadata.json", "w") as f:
    json.dump(image_chunks, f, indent=2)

print("Image metadata saved")


Image metadata saved


In [22]:
query = "comparison table of performance"
results = vectordb.similarity_search(query, k=5)

for i, r in enumerate(results):
    print(f"\nResult {i+1}")
    print("Type:", r.metadata["type"])
    print("Source:", r.metadata["source"])
    print(r.page_content[:300])



Result 1
Type: text
Source: sample.pdf
Fig. 8. Model Performance Comparison Across Evaluation Metrics.

Result 2
Type: table
Source: sample.pdf
Table Columns:
Table 11 | Unnamed: 1 | Unnamed: 2 | Unnamed: 3 | Unnamed: 4

Rows:
Semantic Similarity Metrics between Original and Augmented Speciﬁcations. | nan | nan | nan | nan
Metric | System 1 | System 2 | System 3 | AutoFactory (All)
Average ± Std. Dev | 0.9172 ± 0.0566 | 0.9546 ± 0.0295 | 0.

Result 3
Type: text
Source: sample.pdf
Table 11

Semantic Similarity Metrics between Original and Augmented Specifications.
Metric System 1 System 2 System 3 AutoFactory (All)
Average + Std. Dev 0.9172 + 0.0566 0.9546 + 0.0295 0.9107 + 0.0337 0.9275 + 0.0399
Min / Max Similarity 0.7105 | 0.9879 0.8461 / 0.9924 0.7951 | 0.9849 0.7105 | 0.

Result 4
Type: text
Source: sample.pdf
Table 11

Semantic Similarity Metrics between Original and Augmented Specifications.
Metric System 1 System 2 System 3 AutoFactory (All)
Average + Std. Dev 0.9172 + 0.0566 0